<div align="center">

# 🚀 LaunchMesh · Image to 3D
### Powered by TripoSG · Free · No GPU required on your machine

**How to use:**
1. Click **Runtime → Run all** (or press `Ctrl+F9`)
2. Wait for the app to launch (~3–5 minutes first time)
3. Use the link that appears to open the LaunchMesh UI
4. Upload your image and generate!

---
</div>

In [ ]:
#@title Step 1 - Install dependencies (auto-restarts runtime) { display-mode: "form" }
import subprocess

def run(cmd, label=""):
    if label: print("  -> " + label)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print("  [!] " + r.stderr[-300:])
    return r

print("Installing... (~5 min first time)")

# Clone TripoSG
run("git clone --depth 1 https://github.com/VAST-AI-Research/TripoSG.git /content/TripoSG 2>/dev/null || true", "Cloning TripoSG")

# PyTorch first - diso needs it at build time
run("pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118", "PyTorch")

# diso needs --no-build-isolation to find torch
run("pip install -q diso --no-build-isolation", "diso")

# jaxtyping - required by TripoSG typing utils
run("pip install -q jaxtyping", "jaxtyping")

# Let TripoSG requirements.txt handle diffusers/transformers versions
run("pip install -q -r /content/TripoSG/requirements.txt", "TripoSG requirements")

# Flask + ngrok for the web UI
run("pip install -q flask flask-cors pyngrok", "flask + ngrok")

# Extra utils
run("pip install -q scikit-image jaxtyping", "extra utils")

print("[OK] Done! Restarting runtime...")
print("[!!] After restart: run Steps 2, 3, 4 only - skip Step 1!")

import os
os.kill(os.getpid(), 9)


Installing... (~5 min first time)
  -> Cloning TripoSG
  -> PyTorch
  [!] OR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.0

  -> diso
  -> jaxtyping
  -> TripoSG requirements
  [!] ror originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.

  -> flask + ngrok
  -> extra utils


In [1]:
#@title  Step 2 — Download TripoSG model weights { display-mode: "form" }
from huggingface_hub import snapshot_download
import os

MODEL_DIR = '/content/checkpoints/TripoSG'
RMBG_DIR  = '/content/checkpoints/RMBG-1.4'

if not os.path.exists(MODEL_DIR):
    print('Downloading TripoSG weights (~2 GB)...')
    snapshot_download('VAST-AI/TripoSG', local_dir=MODEL_DIR)
    print('[OK] TripoSG downloaded!')
else:
    print('[OK] TripoSG already downloaded.')

if not os.path.exists(RMBG_DIR):
    print('Downloading background removal model...')
    snapshot_download('briaai/RMBG-1.4', local_dir=RMBG_DIR)
    print('[OK] RMBG downloaded!')
else:
    print('[OK] RMBG already downloaded.')

print('\n All models ready!')


[OK] TripoSG already downloaded.
[OK] RMBG already downloaded.

 All models ready!


In [2]:
#@title  Step 3 — Load models into memory { display-mode: "form" }
import sys, os, torch
sys.path.insert(0, '/content/TripoSG')
sys.path.insert(0, '/content/TripoSG/scripts')

from PIL import Image
import numpy as np
from briarmbg import BriaRMBG
from image_process import prepare_image
from triposg.pipelines.pipeline_triposg import TripoSGPipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16
print(f'Using device: {DEVICE}')

print('Loading background removal model...')
rmbg_net = BriaRMBG.from_pretrained('/content/checkpoints/RMBG-1.4').to(DEVICE)
rmbg_net.eval()

print('Loading TripoSG pipeline...')
pipe = TripoSGPipeline.from_pretrained('/content/checkpoints/TripoSG').to(DEVICE, DTYPE)

print('[OK] Models loaded and ready!')


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Using device: cuda
Loading background removal model...
Loading weights from local directory
Loading TripoSG pipeline...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

[OK] Models loaded and ready!


In [9]:
import subprocess

proc3 = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

for line in proc3.stdout:
    line = line.decode().strip()
    # Look for the line with https:// in it
    if "https://" in line and "trycloudflare.com" in line:
        for word in line.split():
            if word.startswith("https://") and "trycloudflare.com" in word:
                url = word.strip("|").strip()
                print("")
                print("=" * 50)
                print("LaunchMesh is LIVE!")
                print(url)
                print("=" * 50)
                break
        break
    else:
        print(line)  # show progress while waiting

2026-03-20T19:06:20Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-20T19:06:20Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-20T19:06:24Z INF +--------------------------------------------------------------------------------------------+
2026-03-20T19:06:24Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |

LaunchMesh is LIVE!
https://hopes-mix-steve-disc.trycloudflare.com


In [14]:
# Test the fix before putting it in the full cell
import os
test_path = "/content/outputs/test_fix.png"

# Verify a saved image can be processed
from PIL import Image as PILImage
img = PILImage.open(test_path) if os.path.exists(test_path) else None
print("prepare_image expects: file path string")
print("We need to pass: img_path, not the PIL image")

prepare_image expects: file path string
We need to pass: img_path, not the PIL image


In [16]:
# Override prepare_image with a version that handles both paths and PIL images
from PIL import Image as PILImage
import numpy as np
import torch

_original_prepare_image = prepare_image

def prepare_image(image_input, bg_color=[1,1,1], rmbg_net=None):
    # If it's already a PIL image, save it to a temp file first
    if isinstance(image_input, PILImage.Image):
        tmp_path = "/content/outputs/tmp_input.png"
        image_input.save(tmp_path)
        image_input = tmp_path
    return _original_prepare_image(image_input, bg_color, rmbg_net)

print("prepare_image patched!")

prepare_image patched!


In [17]:
# Quick test
import os
# Find any saved image in outputs
imgs = [f for f in os.listdir("/content/outputs") if f.endswith(".png")]
if imgs:
    test = "/content/outputs/" + imgs[0]
    result = prepare_image(test, [1,1,1], rmbg_net)
    print("Works! Result type:", type(result))
else:
    print("No test images yet - upload one through the app first")

TypeError: expected np.ndarray (got list)

In [19]:
import numpy as np
from PIL import Image as PILImage
import importlib, image_process

# Import directly from source module - bypasses our patched version
_true_prepare_image = image_process.prepare_image

def prepare_image(image_input, bg_color=None, rmbg_net=None):
    if bg_color is None or isinstance(bg_color, list):
        bg_color = np.array([1.0, 1.0, 1.0])
    if isinstance(image_input, PILImage.Image):
        tmp_path = "/content/outputs/tmp_input.png"
        image_input.save(tmp_path)
        image_input = tmp_path
    return _true_prepare_image(image_input, bg_color, rmbg_net)

print("prepare_image patched!")

# Test it
import os
imgs = [f for f in os.listdir("/content/outputs") if f.endswith(".png") and "tmp" not in f]
if imgs:
    result = prepare_image("/content/outputs/" + imgs[0])
    print("Works! Result type:", type(result))
else:
    print("No test images - try uploading through the app now")

prepare_image patched!
Works! Result type: <class 'PIL.Image.Image'>


In [ ]:
#@title Step 4 - Launch LaunchMesh App { display-mode: "form" }

import threading, time, os, uuid, subprocess
from flask import Flask, request, jsonify, send_file, Response
from flask_cors import CORS
import trimesh, torch

flask_app = Flask(__name__)
CORS(flask_app)
os.makedirs("/content/outputs", exist_ok=True)
jobs = {}

UI_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>LaunchMesh - Image to 3D</title>
<link href="https://fonts.googleapis.com/css2?family=Rajdhani:wght@500;600;700&family=Exo+2:wght@400;600;800&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{--bg:#040d1a;--surface:#071628;--surface2:#0a1e35;--border:#0e2d50;--orange:#f7931a;--orange2:#ffb347;--blue:#1e90ff;--blue2:#4db8ff;--text:#e8f4ff;--muted:#4a7a9b}
body{background:var(--bg);color:var(--text);font-family:"Exo 2",sans-serif;min-height:100vh;display:flex;flex-direction:column;align-items:center;justify-content:center;padding:40px 20px}
body::before{content:"";position:fixed;inset:0;background:radial-gradient(ellipse at 20% 10%,rgba(30,144,255,.12) 0%,transparent 40%),radial-gradient(ellipse at 80% 90%,rgba(247,147,26,.08) 0%,transparent 40%);pointer-events:none;z-index:0}
.container{position:relative;z-index:1;width:100%;max-width:660px}
header{margin-bottom:36px;text-align:center}
h1{font-family:"Rajdhani",sans-serif;font-size:clamp(2rem,7vw,3.2rem);font-weight:700;letter-spacing:.08em;text-transform:uppercase;line-height:1.1;margin-bottom:10px}
h1 .blue{color:var(--blue2)}h1 .orange{color:var(--orange)}
.subtitle{color:var(--muted);font-size:.82rem;letter-spacing:.18em;text-transform:uppercase;font-weight:600}
.badge{display:inline-block;background:rgba(247,147,26,.15);border:1px solid var(--orange);color:var(--orange);font-size:.65rem;padding:3px 10px;border-radius:20px;letter-spacing:.2em;font-weight:700;text-transform:uppercase;margin-top:10px}
.divider{height:1px;background:linear-gradient(90deg,transparent,var(--border),var(--blue),var(--border),transparent);margin:28px 0}
.drop-zone{border:1.5px dashed var(--border);border-radius:12px;padding:52px 40px;text-align:center;cursor:pointer;transition:all .25s;background:var(--surface);position:relative;overflow:hidden}
.drop-zone:hover,.drop-zone.drag-over{border-color:var(--orange);box-shadow:0 0 30px rgba(247,147,26,.1);transform:translateY(-2px)}
.drop-icon{font-size:2.8rem;margin-bottom:14px;display:block}
.drop-title{font-size:1.05rem;font-weight:700;margin-bottom:6px;font-family:"Rajdhani",sans-serif;letter-spacing:.05em;text-transform:uppercase}
.drop-hint{font-size:.78rem;color:var(--muted);letter-spacing:.1em}
input[type="file"]{display:none}
.preview-wrap{display:none;margin-top:20px;border-radius:10px;overflow:hidden;border:1px solid var(--border);position:relative}
.preview-wrap img{width:100%;max-height:280px;object-fit:contain;background:var(--surface);display:block}
.preview-label{position:absolute;top:10px;left:10px;background:rgba(247,147,26,.9);color:#040d1a;font-size:.65rem;padding:3px 8px;border-radius:4px;letter-spacing:.15em;font-weight:700;text-transform:uppercase}
.btn{display:block;width:100%;margin-top:18px;padding:15px;background:linear-gradient(135deg,var(--orange) 0%,var(--orange2) 100%);color:#040d1a;border:none;border-radius:8px;font-family:"Rajdhani",sans-serif;font-size:1.1rem;font-weight:700;letter-spacing:.12em;text-transform:uppercase;cursor:pointer;transition:all .2s;box-shadow:0 4px 20px rgba(247,147,26,.3)}
.btn:hover{transform:translateY(-2px);box-shadow:0 6px 28px rgba(247,147,26,.45)}
.btn:disabled{opacity:.35;cursor:not-allowed;transform:none;box-shadow:none}
.status-box{display:none;margin-top:20px;padding:18px 22px;background:var(--surface2);border:1px solid var(--border);border-radius:10px}
.status-label{color:var(--muted);font-size:.65rem;letter-spacing:.25em;text-transform:uppercase;margin-bottom:6px;font-weight:600}
.status-text{color:var(--blue2);font-weight:600}
.status-note{color:var(--muted);font-size:.72rem;margin-top:6px}
.progress-bar{height:3px;background:var(--border);border-radius:3px;margin-top:14px;overflow:hidden}
.progress-fill{height:100%;background:linear-gradient(90deg,var(--blue),var(--orange));width:0%;transition:width .5s;border-radius:3px}
.download-btn{display:none;margin-top:18px;padding:15px;background:transparent;color:var(--orange);border:1.5px solid var(--orange);border-radius:8px;font-family:"Rajdhani",sans-serif;font-size:1.1rem;font-weight:700;letter-spacing:.12em;text-transform:uppercase;cursor:pointer;text-align:center;text-decoration:none;transition:all .2s;width:100%}
.download-btn:hover{background:linear-gradient(135deg,var(--orange),var(--orange2));color:#040d1a}
footer{margin-top:36px;text-align:center;color:var(--muted);font-size:.72rem;letter-spacing:.15em;text-transform:uppercase}
footer a{color:var(--blue2);text-decoration:none}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.4}}
.pulsing{animation:pulse 1.5s ease infinite}
</style>
</head>
<body>
<div class="container">
<header>
<h1><span class="blue">Launch</span><span class="orange">Mesh</span></h1>
<p class="subtitle">Image to 3D &middot; Powered by TripoSG</p>
<span class="badge">Free &middot; Community Edition</span>
</header>
<div class="divider"></div>
<div class="drop-zone" id="dz">
<span class="drop-icon">&#128640;</span>
<div class="drop-title">Drop your image here</div>
<div class="drop-hint">or click to browse &middot; PNG &middot; JPG &middot; WEBP</div>
<input type="file" id="fi" accept="image/*">
</div>
<div class="preview-wrap" id="pw"><span class="preview-label">Input</span><img id="pv" src="" alt=""></div>
<button class="btn" id="btn" disabled>&#9889; Generate 3D Model</button>
<div class="status-box" id="sb">
<div class="status-label">Status</div>
<div class="status-text" id="st">Working...</div>
<div class="status-note">TripoSG runs on GPU - takes 1-3 minutes.</div>
<div class="progress-bar"><div class="progress-fill" id="pf"></div></div>
</div>
<a class="download-btn" id="dl" href="#" download="launchmesh_model.glb">&#8681; Download .glb Model</a>
<div class="divider" style="margin-top:32px"></div>
<footer><a href="https://www.skool.com/miless-group-4588" target="_blank">3D Launchpad Community</a></footer>
</div>
<script>
const dz=document.getElementById("dz"),fi=document.getElementById("fi"),pv=document.getElementById("pv"),pw=document.getElementById("pw"),btn=document.getElementById("btn"),sb=document.getElementById("sb"),st=document.getElementById("st"),pf=document.getElementById("pf"),dl=document.getElementById("dl");
let file=null;
dz.onclick=()=>fi.click();
dz.ondragover=e=>{e.preventDefault();dz.classList.add("drag-over")};
dz.ondragleave=()=>dz.classList.remove("drag-over");
dz.ondrop=e=>{e.preventDefault();dz.classList.remove("drag-over");if(e.dataTransfer.files[0])load(e.dataTransfer.files[0])};
fi.onchange=()=>{if(fi.files[0])load(fi.files[0])};
function load(f){file=f;pv.src=URL.createObjectURL(f);pw.style.display="block";btn.disabled=false;dl.style.display="none";sb.style.display="none"}
btn.onclick=async()=>{
  if(!file)return;
  btn.disabled=true;btn.textContent="Generating...";
  sb.style.display="block";pf.style.width="10%";st.textContent="Uploading...";st.classList.add("pulsing");
  const fd=new FormData();
  fd.append("image",file);fd.append("steps","50");fd.append("guidance","7.5");fd.append("seed",String(Math.floor(Math.random()*999999)));
  try{
    const r=await fetch("/generate",{method:"POST",body:fd});
    const d=await r.json();const id=d.job_id;
    pf.style.width="30%";st.textContent="Running TripoSG...";
    let done=false,n=0;
    const labels=["Removing background...","Encoding features...","Running diffusion...","Extracting mesh...","Almost done..."];
    while(!done){
      await new Promise(r=>setTimeout(r,3000));
      const s=await(await fetch("/status/"+id)).json();
      if(s.status==="done"){done=true;pf.style.width="100%";st.textContent="Done!";st.classList.remove("pulsing");dl.href="/download/"+id;dl.style.display="block";btn.textContent="Generate 3D Model";btn.disabled=false;}
      else if(s.status==="error"){throw new Error(s.message||"Failed");}
      else{n++;pf.style.width=Math.min(30+n*10,90)+"%";st.textContent=labels[Math.min(n-1,labels.length-1)];}
    }
  }catch(e){st.textContent="Error: "+e.message;st.classList.remove("pulsing");st.style.color="#f87171";btn.textContent="Generate 3D Model";btn.disabled=false;}
};
</script>
</body>
</html>"""

@flask_app.route("/")
def index():
    return Response(UI_HTML, mimetype="text/html")

@flask_app.route("/generate", methods=["POST"])
def generate():
    import random
    f = request.files["image"]
    job_id = str(uuid.uuid4())[:8]
    img_path = "/content/outputs/" + job_id + ".png"
    out_path = "/content/outputs/" + job_id + ".glb"
    f.save(img_path)
    jobs[job_id] = {"status": "running"}

    # Read form values BEFORE entering thread
    _seed     = int(request.form.get("seed", random.randint(0, 999999)))
    _steps    = int(request.form.get("steps", 50))
    _guidance = float(request.form.get("guidance", 7.5))

    def run_job():
        try:
            # Pass file PATH to prepare_image, not a PIL object
            processed = prepare_image(img_path, [1, 1, 1], rmbg_net)
            torch.manual_seed(_seed)
            out = pipe(
                image=processed,
                num_inference_steps=_steps,
                guidance_scale=_guidance
            ).samples[0]
            verts, faces = out
            mesh = trimesh.Trimesh(verts.cpu().numpy(), faces.cpu().numpy())
            mesh.export(out_path)
            jobs[job_id]["status"] = "done"
            jobs[job_id]["out_path"] = out_path
            print("[OK] Job done:", job_id)
        except Exception as e:
            jobs[job_id] = {"status": "error", "message": str(e)}
            print("[ERROR]", e)

    threading.Thread(target=run_job, daemon=True).start()
    return jsonify({"job_id": job_id})

@flask_app.route("/status/<job_id>")
def status(job_id):
    return jsonify(jobs.get(job_id, {"status": "unknown"}))

@flask_app.route("/download/<job_id>")
def download(job_id):
    job = jobs.get(job_id)
    if not job or job["status"] != "done":
        return "Not ready", 404
    return send_file(job["out_path"], as_attachment=True, download_name="launchmesh_model.glb")

# Install cloudflared
print("Installing cloudflared...")
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True
)

# Start Flask in background thread
threading.Thread(
    target=lambda: flask_app.run(host="0.0.0.0", port=5000, use_reloader=False, debug=False),
    daemon=True
).start()
time.sleep(2)
print("Flask running!")

# Start cloudflared and grab URL
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print("Starting tunnel - waiting for URL...")
for line in proc.stdout:
    line = line.decode().strip()
    if "https://" in line and "trycloudflare.com" in line:
        for word in line.split():
            if word.startswith("https://") and "trycloudflare.com" in word:
                url = word.strip("|").strip()
                print("")
                print("=" * 50)
                print("LaunchMesh is LIVE!")
                print(url)
                print("=" * 50)
                print("Share this with your 3D community!")
                print("Keep this cell running to stay online.")
                break
        break

proc.wait()

Installing cloudflared...
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Flask running!
Starting tunnel - waiting for URL...

LaunchMesh is LIVE!
https://careful-assistance-decade-unable.trycloudflare.com
Share this with your 3D community!
Keep this cell running to stay online.


---
## 💡 Tips for best results
- **Clean background** — white or solid colour behind the object
- **Single object** — one item centred in frame
- **Good lighting** — evenly lit, minimal shadows
- **Square crop** — crop tight around the object

## 📁 Output
Downloads as a `.glb` file — open in **Blender** (File → Import → glTF 2.0), Windows 3D Viewer, Unity, or Unreal.

## ⚡ Free GPU limits
Each person who opens their own copy of this notebook gets their own independent Colab GPU session — no shared quota!

## 🔗 Your shareable link
```
https://colab.research.google.com/github/chopsitv/launchmesh-triposg/blob/main/LaunchMesh_TripoSG.ipynb
```